# Neural Networks for Unsupervised Learning -- Autoencoders

In this exercise we build an **autoencoder** -- a neural network that learns to compress data into a low-dimensional latent representation and then reconstruct it.

**Dataset:** `classif_train.csv` (10,000 samples) and `classif_test.csv` (1,000 samples) with columns `x`, `y`, and `label`. The points are spread uniformly over a square, and `label` marks the points outside a circle of radius 3. The labels are not used for training, only to colour some plots.

**Evaluation protocol:** we hold out 20% of the training file (2,000 rows) as a **validation set**. It monitors training and sets the anomaly threshold. The test file is used **once**, at the end, to report how many rows the fixed threshold flags. The scaler is fitted on the training rows only.

**Learning objectives:**
1. Build an encoder-decoder architecture in PyTorch.
2. Train on reconstruction loss (MSE).
3. Visualize original vs. reconstructed samples.
4. Extract and visualize latent representations.
5. Use reconstruction error as an anomaly score, with a threshold set on validation data.

---
## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE

SEED = 42
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__}, device: {DEVICE}")

---
## 2. Load and Explore the Data

Load `classif_train.csv` and `classif_test.csv`. The first column is the index. Columns are `x`, `y`, and `label`. Explore the training file only: the test file is kept for the final check in section 8.

In [ ]:
# YOUR CODE HERE
# Hint:
#   df_train = pd.read_csv("classif_train.csv", index_col=0)
#   df_test  = pd.read_csv("classif_test.csv",  index_col=0)


In [ ]:
# YOUR CODE HERE
# Hint: scatter plot of x vs y for the training file, coloured by label

---
## 3. Prepare Data for PyTorch

We hold out 20% of the training file as validation rows (seeded split) **before** fitting the scaler, and fit the scaler on the remaining training rows only. The validation rows monitor training (section 5) and set the anomaly threshold (section 8); the test rows are transformed with the same scaler but not used until section 8.

For an autoencoder, the **target is the input itself**: `TensorDataset(X, X)`.

**Tasks:**
- Extract the features (`x`, `y`) and hold out 20% of the training rows with `train_test_split(..., test_size=0.2, random_state=SEED)`
- Fit `StandardScaler` on the training rows only; transform training, validation, and test rows
- Convert to tensors and build `TensorDataset(X, X)` loaders for the training and validation rows

In [ ]:
# YOUR CODE HERE
# Hint:
#   X_all = df_train[["x", "y"]].values.astype(np.float32)
#   X_train, X_val, labels_train, labels_val = train_test_split(
#       X_all, df_train["label"].values, test_size=0.2, random_state=SEED)
#   X_test = df_test[["x", "y"]].values.astype(np.float32)   # used once, section 8
#   scaler = StandardScaler().fit(X_train)                    # training rows only
#   X_train_sc = scaler.transform(X_train).astype(np.float32)  # same for X_val_sc, X_test_sc
#   train_tensor = torch.tensor(X_train_sc)                     # same for val_tensor, test_tensor
#   train_ds = TensorDataset(train_tensor, train_tensor)        # input = target
#   train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
#   val_loader = DataLoader(TensorDataset(val_tensor, val_tensor), batch_size=64, shuffle=False)

---
## 4. Build the Autoencoder

Create a class `Autoencoder(nn.Module)` with:
- **Encoder:** Linear(2->16) -> ReLU -> Linear(16->8) -> ReLU -> Linear(8->latent_dim)
- **Decoder:** Linear(latent_dim->8) -> ReLU -> Linear(8->16) -> ReLU -> Linear(16->2)
- `forward(x)` returns the reconstruction
- `encode(x)` returns the latent representation

Use `latent_dim=1` (a one-number bottleneck) and call `torch.manual_seed(SEED)` before creating the model.

In [ ]:
# YOUR CODE HERE
# Hint: use nn.Sequential for both self.encoder and self.decoder
# class Autoencoder(nn.Module):
#     def __init__(self, input_dim=2, latent_dim=1):
#         ...
#     def forward(self, x): return self.decoder(self.encoder(x))
#     def encode(self, x): return self.encoder(x)
# torch.manual_seed(SEED); model = Autoencoder(input_dim=2, latent_dim=1).to(DEVICE)

---
## 5. Train the Autoencoder

Use `nn.MSELoss()` and `optim.Adam(lr=1e-3)`. Train for 100 epochs.
Record the training loss and the **validation** loss (on `val_loader`, not the test file) per epoch.

In [ ]:
# YOUR CODE HERE
# Hint: standard training loop
# For each batch: optimizer.zero_grad(), x_hat = model(x_batch), loss = criterion(x_hat, x_batch)
# loss.backward(), optimizer.step()
# After each epoch: model.eval() and the validation loss on val_loader (not the test file)

In [ ]:
# YOUR CODE HERE
# Hint: plot train_losses and val_losses


---
## 6. Visualize Original vs. Reconstructed Samples (validation rows)

Run the validation rows through the autoencoder, inverse-transform with the scaler, and compare scatter plots.

In [ ]:
# YOUR CODE HERE
# Hint:
#   model.eval()
#   with torch.no_grad():
#       X_val_hat = model(val_tensor.to(DEVICE)).cpu().numpy()
#   X_val_recon = scaler.inverse_transform(X_val_hat)
#   Plot original and reconstructed validation rows side by side

---
## 7. Extract and Visualize Latent Representations

Use `model.encode()` to get latent vectors. If latent_dim=1, plot a histogram coloured by label.
If latent_dim>1, use t-SNE to project to 2D.

In [ ]:
# YOUR CODE HERE
# Hint:
#   with torch.no_grad():
#       z_train = model.encode(train_tensor.to(DEVICE)).cpu().numpy()
#   For 1D: plt.hist for each label
#   For higher dim: TSNE(n_components=2).fit_transform(z_train[:2000])


In [ ]:
# YOUR CODE HERE
# Visualization of the latent space


---
## 8. Anomaly Detection via Reconstruction Error

Points that the autoencoder struggles to reconstruct may be anomalies (far from the learned data manifold). The threshold is a **decision**: we take the 95th percentile of the reconstruction errors on the **validation** rows, so about 5% of normal-like rows will be flagged by construction. A percentile sets an alarm rate; it does not measure accuracy. We fix the threshold on the validation rows, then apply it **once** to the test file.

**Tasks:**
- Compute the per-row MSE between the validation rows and their reconstructions, and set `threshold = np.percentile(val_error, 95)`
- Plot the histogram of validation errors with the threshold
- Then, once: compute the test errors, flag `test_error > threshold`, report the flagged fraction, and plot where the flagged test points lie

In [ ]:
# YOUR CODE HERE
# Hint:
#   val_error = np.mean((X_val_sc - X_val_hat) ** 2, axis=1)
#   threshold = np.percentile(val_error, 95)        # chosen on validation rows
#   Plot the histogram of val_error with the threshold as a vertical line

In [ ]:
# YOUR CODE HERE
# Apply the fixed threshold once to the test file
# Hint: test_error = np.mean((X_test_sc - X_test_hat) ** 2, axis=1); anomalies = test_error > threshold
# Scatter plot: rows below the threshold in blue, flagged rows in red

**Your interpretation:** Where do the flagged points lie, and why there? What does the flagged fraction on the test file tell you, and what can it not tell you?

---
## 9. Summary

In this notebook we:

1. Loaded a 2D dataset and visualized the training file.
2. Held out validation rows from the training file and fitted the scaler on the training rows only.
3. Built a simple **autoencoder** (encoder-decoder) in PyTorch and trained it on MSE reconstruction loss, monitoring the validation rows.
4. Visualized original vs. reconstructed validation points.
5. Extracted latent representations and visualized their distribution.
6. Set an anomaly threshold on the validation rows (95th percentile of their reconstruction errors) and applied it once to the test file.

**Key takeaways:**
- Autoencoders learn a compressed representation of the data without labels.
- The latent space captures the most important structure of the data.
- Reconstruction error is a natural anomaly score -- samples that are hard to reconstruct lie far from the learned manifold.
- A percentile threshold sets the alarm rate: choose it on validation data, report it on test data, and do not read a flag as a diagnosis.